In [1]:
import json
import os
import sys
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# VS Code 运行 notebook 时，当前目录是 notebooks/；这里找到项目根目录
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.append(str(ROOT / "src"))
from fetch_utils import get_json  # 成员 E 写的共用取数函数

RAW = ROOT / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

START = "2023-01-01"
END = "2026-10-31"  # 数据冻结日


def save_json(data, filename):
    """把 API 返回的原始内容原样存进 data/raw/。"""
    path = RAW / filename
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"  已保存 {path.relative_to(ROOT)}")


print("项目根目录:", ROOT)
print("原始数据文件夹:", RAW.relative_to(ROOT))

项目根目录: /Users/rhea/Desktop/northbound_project
原始数据文件夹: data/raw


In [2]:
FRANKFURTER = "https://api.frankfurter.dev/v1"


def frankfurter_url(year):
    """某一年的时间序列网址：第一年从 START 开始，最后一年只到冻结日。"""
    start = START if year == 2023 else f"{year}-01-01"
    end = END if year == 2026 else f"{year}-12-31"
    return f"{FRANKFURTER}/{start}..{end}"


fx_rows = []
for year in [2023, 2024, 2025, 2026]:
    data = get_json(frankfurter_url(year), params={"base": "HKD", "symbols": "CNY"})
    if data is None:
        print(f"{year} 年下载失败：检查网络后重跑这一格")
        continue
    save_json(data, f"frankfurter_hkd_cny_{year}.json")
    for date, rates in data["rates"].items():
        fx_rows.append({"date": date, "hkd_cny": rates["CNY"]})
    print(f"{year}: {len(data['rates'])} 个工作日")
    time.sleep(1)

  已保存 data/raw/frankfurter_hkd_cny_2023.json
2023: 256 个工作日
  已保存 data/raw/frankfurter_hkd_cny_2024.json
2024: 257 个工作日
  已保存 data/raw/frankfurter_hkd_cny_2025.json
2025: 256 个工作日
  已保存 data/raw/frankfurter_hkd_cny_2026.json
2026: 214 个工作日


In [3]:
# 检查：结构和数字是否合理
fx = pd.DataFrame(fx_rows)
fx["date"] = pd.to_datetime(fx["date"])
fx = fx.sort_values("date")

# 每年从 1 月 1 日开始请求时，Frankfurter 会多返回上一年最后一个工作日
print("清理前行数:", len(fx))
print("重复的日期:", fx.loc[fx["date"].duplicated(), "date"].dt.date.tolist())
fx = fx.drop_duplicates("date")
fx = fx[(fx["date"] >= START) & (fx["date"] <= END)]
print("清理后行数:", len(fx))

print("\n日期范围:", fx["date"].min().date(), "至", fx["date"].max().date())
print("重复日期:", fx["date"].duplicated().sum())
print("汇率范围:", fx["hkd_cny"].min(), "至", fx["hkd_cny"].max(), "（应在 0.8–1.0 之间）")
print(fx.head())

fx.to_csv(RAW / "fx_frankfurter_daily.csv", index=False)
print("\n已保存 data/raw/fx_frankfurter_daily.csv")


清理前行数: 983
重复的日期: [datetime.date(2023, 12, 29), datetime.date(2024, 12, 31), datetime.date(2025, 12, 31)]
清理后行数: 979

日期范围: 2023-01-02 至 2026-10-30
重复日期: 0
汇率范围: 0.84797 至 0.94785 （应在 0.8–1.0 之间）
        date  hkd_cny
1 2023-01-02  0.88379
2 2023-01-03  0.88435
3 2023-01-04  0.88078
4 2023-01-05  0.88012
5 2023-01-06  0.87853

已保存 data/raw/fx_frankfurter_daily.csv


In [4]:
load_dotenv(ROOT / ".env")
FRED_KEY = os.getenv("FRED_API_KEY")
print("读到 FRED 密钥:", FRED_KEY is not None, "| 长度:", len(FRED_KEY or ""), "（应为 32）")

FRED_URL = "https://api.stlouisfed.org/fred/series/observations"
FRED_SERIES = {"DEXCHUS": "cny_per_usd", "DEXHKUS": "hkd_per_usd"}

fred_tables = []
for series_id, column in FRED_SERIES.items():
    params = {
        "series_id": series_id,
        "api_key": FRED_KEY,
        "file_type": "json",
        "observation_start": START,
        "observation_end": END,
    }
    data = get_json(FRED_URL, params=params)
    if data is None:
        print(f"{series_id} 下载失败：检查 .env 里的密钥和网络")
        continue
    save_json(data, f"fred_{series_id}.json")  # FRED 的返回内容里不含密钥
    obs = pd.DataFrame(data["observations"])
    obs[column] = pd.to_numeric(obs["value"], errors="coerce")
    fred_tables.append(obs[["date", column]].set_index("date"))
    print(f"{series_id}: {len(obs)} 行，其中缺值 {obs[column].isna().sum()} 个")
    time.sleep(1)

读到 FRED 密钥: True | 长度: 32 （应为 32）
  已保存 data/raw/fred_DEXCHUS.json
DEXCHUS: 1000 行，其中缺值 41 个
  已保存 data/raw/fred_DEXHKUS.json
DEXHKUS: 1000 行，其中缺值 41 个


In [5]:
# 检查：合并两个序列，换算成港币兑人民币，再和 Frankfurter 同一天对比
fred = pd.concat(fred_tables, axis=1).reset_index()
fred["date"] = pd.to_datetime(fred["date"])
fred["fred_hkd_cny"] = fred["cny_per_usd"] / fred["hkd_per_usd"]

print("日期范围:", fred["date"].min().date(), "至", fred["date"].max().date())
print("重复日期:", fred["date"].duplicated().sum())

both = fx.merge(fred[["date", "fred_hkd_cny"]], on="date").dropna()
gap = (both["fred_hkd_cny"] / both["hkd_cny"] - 1).abs() * 100
print("\n两个来源同一天都有数据:", len(both), "天")
print(f"差距：平均 {gap.mean():.3f}%，最大 {gap.max():.3f}%（平均应小于 0.5%）")

fred.to_csv(RAW / "fx_fred_daily.csv", index=False)
print("\n已保存 data/raw/fx_fred_daily.csv")


日期范围: 2023-01-02 至 2026-10-30
重复日期: 0

两个来源同一天都有数据: 944 天
差距：平均 0.051%，最大 0.709%（平均应小于 0.5%）

已保存 data/raw/fx_fred_daily.csv


In [6]:
import yfinance as yf

hsi = yf.Ticker("^HSI").history(start=START, end="2026-11-01", auto_adjust=False)

if hsi.empty:
    print("没取到数据：Yahoo 可能暂时限制了访问，过几分钟再重跑这一格")
else:
    hsi.index = hsi.index.tz_localize(None).normalize()  # 去掉时区，只保留日期
    hsi = hsi[["Open", "High", "Low", "Close", "Volume"]].reset_index()
    hsi = hsi.rename(columns={"Date": "date"})

    print("行数:", len(hsi), "（香港交易日，应在 920–960 之间）")
    print("日期范围:", hsi["date"].min().date(), "至", hsi["date"].max().date())
    print("收盘价范围:", round(hsi["Close"].min()), "至", round(hsi["Close"].max()))
    print(hsi.head())

    hsi.to_csv(RAW / "hsi_daily.csv", index=False)
    print("\n已保存 data/raw/hsi_daily.csv")

行数: 939 （香港交易日，应在 920–960 之间）
日期范围: 2023-01-03 至 2026-10-30
收盘价范围: 14961 至 27968
        date          Open          High           Low         Close  \
0 2023-01-03  19570.429688  20212.550781  19303.730469  20145.289062   
1 2023-01-04  20319.980469  20793.109375  20233.390625  20793.109375   
2 2023-01-05  21295.869141  21396.089844  20962.400391  21052.169922   
3 2023-01-06  21220.890625  21282.710938  20862.769531  20991.640625   
4 2023-01-09  21295.939453  21470.689453  21216.919922  21388.339844   

       Volume  
0  2286700000  
1  3016000000  
2  2875400000  
3  2754500000  
4  2697900000  

已保存 data/raw/hsi_daily.csv


/Users/rhea/Desktop/northbound_project/.venv/lib/python3.14/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)


In [7]:
import requests

CSD_URL = "https://www.censtatd.gov.hk/api/get.php"
CSD_PARAMS = {"id": "620-67001", "lang": "en", "full_series": "1"}
# 统计处会拒绝 Python 默认的身份（403），这里按统计处 API 指南的示例设置
CSD_HEADERS = {"User-Agent": "Python Request"}

try:
    response = requests.get(CSD_URL, params=CSD_PARAMS, headers=CSD_HEADERS, timeout=60)
    response.raise_for_status()
    data = response.json()
except (requests.exceptions.RequestException, ValueError) as error:
    print("下载失败:", type(error).__name__)
    data = None

if data is not None:
    save_json(data, "csd_620-67001.json")
    print("最外层的键:", list(data.keys()))

    retail_raw = pd.DataFrame(data["dataSet"])
    print("\n行数:", len(retail_raw))
    print("列名:", retail_raw.columns.tolist())
    print("\n前 5 行:")
    print(retail_raw.head().to_string())

    print("\n文字列各有哪些取值（用来判断怎么筛选）:")
    for col in retail_raw.columns:
        values = retail_raw[col].astype(str).unique()
        if len(values) <= 15:
            print(f"  {col}: {list(values)}")

  已保存 data/raw/csd_620-67001.json
最外层的键: ['header', 'dataSet']

行数: 1430
列名: ['freq', 'period', 'sv', 'svDesc', 'figure', 'sd_value']

前 5 行:
  freq  period      sv       svDesc  figure sd_value
0    Y    2004  VAL_RS  HK$ million             N.A.
1    M  200410  VAL_RS  HK$ million   15831         
2    M  200411  VAL_RS  HK$ million   15134         
3    M  200412  VAL_RS  HK$ million   18025         
4    Y    2005  VAL_RS  HK$ million  204372         

文字列各有哪些取值（用来判断怎么筛选）:
  freq: ['Y', 'M']
  sv: ['VAL_RS', 'VOL_IDX_RS', 'VAL_IDX_RS']
  svDesc: ['HK$ million', 'Index', 'Year-on-year % change']
  sd_value: ['N.A.', '', 'p']


In [8]:
# 只要：月度 + 销货价值 + 单位是百万港元（同一个 VAL_RS 下还有"按年变动 %"，要排除）
retail = retail_raw[
    (retail_raw["freq"] == "M")
    & (retail_raw["sv"] == "VAL_RS")
    & (retail_raw["svDesc"] == "HK$ million")
].copy()

retail["month"] = pd.to_datetime(retail["period"], format="%Y%m")
retail["retail_value_hkd_mn"] = pd.to_numeric(retail["figure"], errors="coerce")
retail["provisional"] = retail["sd_value"] == "p"  # p = 临时数字，之后可能修订
retail = retail[retail["month"] >= START]
retail = retail[["month", "retail_value_hkd_mn", "provisional"]].sort_values("month")

print("行数:", len(retail), "（2023-01 起每月一行，应在 44–46 之间）")
print("月份范围:", retail["month"].min().strftime("%Y-%m"), "至", retail["month"].max().strftime("%Y-%m"))
print("重复月份:", retail["month"].duplicated().sum())
print("缺值:", retail["retail_value_hkd_mn"].isna().sum())
print("销货价值范围（百万港元）:", retail["retail_value_hkd_mn"].min(), "至", retail["retail_value_hkd_mn"].max())
print("临时数字的月份:", retail.loc[retail["provisional"], "month"].dt.strftime("%Y-%m").tolist())
print(retail.tail())

retail.to_csv(RAW / "csd_retail_monthly.csv", index=False)
print("\n已保存 data/raw/csd_retail_monthly.csv")


行数: 45 （2023-01 起每月一行，应在 44–46 之间）
月份范围: 2023-01 至 2026-09
重复月份: 0
缺值: 0
销货价值范围（百万港元）: 28876 至 37327
临时数字的月份: ['2026-09']
         month  retail_value_hkd_mn  provisional
281 2026-05-01                33790        False
282 2026-06-01                31490        False
283 2026-07-01                31020        False
284 2026-08-01                32076        False
285 2026-09-01                33357         True

已保存 data/raw/csd_retail_monthly.csv
